# Fischer–Tropsch LTFT

Triagem heurística não calibrada. Requer numpy, pandas, openpyxl, matminer e pymatgen. Alpha pode ser estimado pelos priors auditáveis ou informado. Não calcula conversão de CO nem produtividade.


In [ ]:
metais = ["Co", "Fe"]
promotor = "K"
temperatura_C = 225
pressao_bar = 20
razao_H2_CO = 2.0
alpha_informado = None
pasta_saida = "resultados_ltft"


In [ ]:
"""Ideal ASF distribution, conditional on hydrocarbon formation.

alpha is an input, not a prediction from catalyst or operating conditions.
Carbon fractions n*(1-alpha)^2*alpha**(n-1) are not exact paraffin mass fractions.
"""
import math
from numbers import Integral, Real


def validate_alpha(alpha):
    if isinstance(alpha, bool) or not isinstance(alpha, Real) or not math.isfinite(alpha) or not 0 <= alpha < 1:
        raise ValueError("alpha must be finite and satisfy 0 <= alpha < 1")
    return float(alpha)


def _integer(n):
    if isinstance(n, bool) or not isinstance(n, Integral) or n < 1:
        raise ValueError("Carbon number must be a positive integer")


def tail_fraction(alpha, start, basis="carbon"):
    """Exact infinite tail, including start; no finite-grid renormalization."""
    a = validate_alpha(alpha)
    _integer(start)
    if basis not in {"carbon", "molar", "paraffin_mass"}:
        raise ValueError("Unknown fraction basis")
    molar = a ** (start - 1)
    carbon = molar * (1 + (start - 1) * (1 - a))
    if basis == "molar":
        return molar
    if basis == "carbon":
        return carbon
    # CnH(2n+2): M(n) = (MC+2MH)*n + 2MH.
    unit, ends = 12.011 + 2 * 1.008, 2 * 1.008
    return (unit * carbon + ends * (1-a) * molar) / (unit + ends * (1-a))


def band_fraction(alpha, start, end=None, basis="carbon"):
    _integer(start)
    if end is not None:
        _integer(end)
        if end < start:
            raise ValueError("end must be >= start")
    return max(0.0, tail_fraction(alpha, start, basis) - (tail_fraction(alpha, end+1, basis) if end is not None else 0.0))


def distribution(alpha, max_carbon=60, basis="carbon"):
    """Return finite plotting rows and an explicit remainder through infinity."""
    a = validate_alpha(alpha)
    _integer(max_carbon)
    if max_carbon > 10000:
        raise ValueError("max_carbon must not exceed 10000")
    rows = [{"carbon_number": n, "fraction": band_fraction(a, n, n, basis)} for n in range(1, max_carbon+1)]
    groups = {name: band_fraction(a, low, high, basis) for name, low, high in
              [("CH4", 1, 1), ("C2-C4", 2, 4), ("C5-C11", 5, 11),
               ("C12-C20", 12, 20), ("C21+", 21, None)]}
    return {"alpha": a, "basis": basis, "model": "ideal_ASF_user_supplied_alpha",
            "rows": rows, "tail_start": max_carbon+1,
            "tail_fraction": tail_fraction(a, max_carbon+1, basis),
            "exclusive_groups": groups, "C5plus_subtotal": tail_fraction(a, 5, basis),
            "closure_error": abs(math.fsum(groups.values())-1.0)}


def aviation_screen(alpha, jet_min=8, jet_max=16, basis="carbon"):
    """Configurable carbon-number cut, not aviation-fuel qualification."""
    _integer(jet_min)
    _integer(jet_max)
    if jet_min < 2 or jet_max < jet_min:
        raise ValueError("Require 2 <= jet_min <= jet_max")
    return {"alpha": validate_alpha(alpha), "basis": basis,
            "cut_definition": f"C{jet_min}-C{jet_max} (screening convention)",
            "lighter_fraction": band_fraction(alpha, 1, jet_min-1, basis),
            "direct_cut_fraction": band_fraction(alpha, jet_min, jet_max, basis),
            "heavy_feed_fraction": tail_fraction(alpha, jet_max+1, basis),
            "upgraded_jet_yield": None, "fuel_qualification": "not_assessed"}


In [ ]:
"""Evidence-aware exports. Scores are not experimental probabilities or yields."""
import json
import re
import unicodedata

import numpy as np
import pandas as pd


def output_prefix(reaction, metals, promoter):
    def slug(value):
        value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
        return re.sub(r"[^a-zA-Z0-9_-]+", "-", value).strip("-")
    reaction = {"reforma": "reforma-CH4-CO2", "metanacao": "metanacao-CO2", "rwgs": "RWGS-CO2"}.get(reaction, reaction)
    promoter = "sem-promotor" if not promoter else "promotor-" + slug(promoter)
    return "catailab_" + slug(reaction) + "_" + "-".join(map(slug, metals)) + "_" + promoter


def audited_export(frame, reaction):
    """Keep internal numerical scores intact; expose their actual evidence limits."""
    result = frame.copy()
    if "formula" not in result:
        return result
    if "score_incerteza" in result:
        result["indice_evidencia_interno_0_1"] = result.pop("score_incerteza")
    if "confiabilidade" in result:
        result["classe_indice_interno_nao_calibrado"] = result.pop("confiabilidade")
    result["validacao_experimental"] = "Não disponível nesta execução"
    domain = result.get("classe_dominio_aplicabilidade", pd.Series("nao_avaliado", index=result.index))
    result["aviso_dominio"] = domain.map(lambda x: "Extrapolação: não interpretar como previsão validada" if str(x) == "fora_do_dominio" else "Domínio de referência não equivale a validação experimental")
    if "fonte_estabilidade_triagem" in result:
        result["fonte_estabilidade_original"] = result["fonte_estabilidade_triagem"]
        model = result.get("modelo_gnn_local", pd.Series("", index=result.index)).fillna("").astype(str)
        energy = pd.to_numeric(result.get("energia_gnn_eV_atom", pd.Series(np.nan, index=result.index)), errors="coerce")
        used = result.get("gnn_local_usado", pd.Series(False, index=result.index)).map(lambda x: x is True or str(x).lower() in ("true", "1"))
        valid = used & model.str.strip().ne("") & np.isfinite(energy)
        claimed = result["fonte_estabilidade_triagem"].astype(str).str.contains("GNN", case=False, na=False)
        result.loc[claimed & ~valid, "fonte_estabilidade_triagem"] = "Estimativa heurística; execução GNN não comprovada"
        result["gnn_execucao_comprovada"] = valid
        result["versao_gnn_registrada"] = result.get("versao_gnn_local", "Não registrada")
    if "suporte_sugerido" in result:
        result["alternativas_suporte"] = result["suporte_sugerido"].fillna("").astype(str)
        ambiguous = result["alternativas_suporte"].str.contains(r"\s+ou\s+", regex=True)
        # Preserve the proposed alternatives; status records that none was selected.
        result.loc[ambiguous, "suporte_sugerido"] = result.loc[ambiguous, "alternativas_suporte"]
        result["formulacao_status"] = np.where(ambiguous, "Incompleta: suporte e cargas devem ser definidos", "Carga total e preparação precisam de confirmação")
    result["base_formula"] = "Proporção atômica; não representa teor em massa do catalisador suportado"
    result["teores_massa_catalisador_final"] = "Não definidos nesta triagem; não inferir da fórmula atômica"
    replacements = {
        "score_resistencia_coque": "indice_resistencia_coque_composicional_0_1",
        "taxa_desativacao_coque_proxy": "indice_desativacao_coque_proxy_sem_calibracao",
        "taxa_desativacao_coque_condicao": "indice_desativacao_condicional_sem_calibracao",
        "conversao_prevista_pct": "conversao_proxy_nao_calibrada_pct",
        "seletividade_produto_prevista_pct": "seletividade_proxy_nao_calibrada_pct",
        "rendimento_ou_produtividade_prevista_pct": "indice_rendimento_proxy_pct",
    }
    result = result.rename(columns=replacements)
    result["interpretacao_coque"] = "Índice composicional e penalidade operacional têm bases distintas; não são massa de carbono nem vida útil"
    result["definicao_indice_rendimento"] = "Conversão proxy x seletividade proxy / 100; não é produtividade nem rendimento de H2 validado"
    if reaction == "reforma":
        for col in ("conversao_CH4_validada_pct", "conversao_CO2_validada_pct", "rendimento_H2_validado_pct", "razao_H2_CO_validada"):
            result[col] = np.nan
    result["produtividade_fisica"] = np.nan
    return result


EXPORT_LABELS = {
    "score_final": "Score final de priorização (0–1; não é probabilidade)",
    "indice_evidencia_interno_0_1": "Índice interno de evidência (não é probabilidade)",
    "probabilidade_top5_mc": "Frequência no Top 5 sob perturbações Monte Carlo (0–1)",
    "conversao_proxy_nao_calibrada_pct": "Conversão proxy não calibrada (%)",
    "seletividade_proxy_nao_calibrada_pct": "Seletividade proxy não calibrada (%)",
    "indice_rendimento_proxy_pct": "Índice de rendimento proxy (%)",
    "indice_resistencia_coque_composicional_0_1": "Resistência composicional ao coque (índice 0–1)",
}

REPORT_NOTICE = """<section><h2>Limites científicos e definição dos resultados</h2>
<p>A pontuação multicritério prioriza candidatos; não é probabilidade de acerto.
O Monte Carlo mede estabilidade do ranking sob as perturbações assumidas, não validação experimental.
Candidatos fora do domínio são extrapolações. Os indicadores de conversão, seletividade e rendimento
são proxies não calibrados. Não há produtividade física, rendimento de H₂ validado ou tempo de vida
experimental calculados nesta execução. Campos vazios significam não calculado, nunca zero.</p>
<p>As fórmulas indicam proporções atômicas. Suportes alternativos não constituem um único material.
Antes da síntese, definir suporte, carga total, precursores, pureza e tratamentos térmicos.
Os índices de coque composicional e operacional não devem ser confundidos com massa de carbono.</p></section>"""


def support_alternatives(frame):
    records = []
    if "formula" not in frame or "suporte_sugerido" not in frame:
        return pd.DataFrame()
    for _, row in frame.iterrows():
        options = re.split(r",\s*|\s+ou\s+", str(row["suporte_sugerido"]))
        for i, support in enumerate(options, 1):
            records.append({"candidato": row["formula"], "alternativa": i,
                "suporte": support.strip(), "status": "Hipótese de formulação; desempenho não recalculado por suporte",
                "carga_metalica_pct_massa": None, "carga_promotor_pct_massa": None,
                "rota_proposta": row.get("rota_sintese_sugerida", "Não definida"),
                "tratamento_proposto": row.get("pretratamento_sugerido", "Não definido")})
    return pd.DataFrame(records).drop_duplicates()


In [ ]:
"""LTFT screening: explicit heuristic priors, not calibrated kinetics."""
import hashlib
import html
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_VERSION = 'ltft-heuristic-2'
CHEMICAL_PROFILES = {
    'Co': {
        'activation': 'Avaliar redução dos precursores para Co metálico; confirmar fase e dispersão.',
        'wgs_status': 'Baixa atividade WGS como hipótese qualitativa; não assumir taxa zero.',
        'phase_warning': 'Água, reoxidação e interação com suporte não são resolvidas pelo modelo.',
    },
    'Fe': {
        'activation': 'Avaliar ativação e carburização; confirmar carbetos e óxidos coexistentes.',
        'wgs_status': 'WGS relevante: CO + H2O ⇌ CO2 + H2. Extensão e taxas não calculadas.',
        'phase_warning': 'Distribuição de fases depende da ativação e do ambiente; não presumir um carbeto único.',
    },
    'Co-Fe exploratório': {
        'activation': 'Avaliar redução e carburização de forma conjunta; caracterizar as fases presentes.',
        'wgs_status': 'WGS possível pela presença de Fe; não interpolar taxas entre Co e Fe.',
        'phase_warning': 'Mistura exploratória: não comprova liga, sinergia ou fases ativas coexistentes.',
    },
}
# Engineering priors, not fitted parameters or measured catalyst properties.
PRIORS = {
    'Co': {'alpha': .86, 'activity': .75, 'phase': 'Co0 (hipótese)', 'phase_score': .75},
    'Fe': {'alpha': .82, 'activity': .65, 'phase': 'Carbetos de Fe (hipótese)', 'phase_score': .60},
}
SUPPORTS = {'SiO2': .75, 'Al2O3': .70, 'TiO2': .80, 'ZrO2': .70, 'C': .65}
PROMOTERS = ('K', 'Na', 'Mn', 'Cu', 'Ru', 'Re', 'La', 'Ce', 'Zr')
WEIGHTS = {'activity': .25, 'C5plus': .25, 'growth': .15, 'phase': .15, 'stability': .10, 'operation': .10}


def validate_config(metals, promoter, temperature, pressure, ratio):
    if not metals or len(set(metals)) != len(metals) or not set(metals) <= {'Co', 'Fe'}:
        raise ValueError('LTFT: selecione Co, Fe ou Co e Fe. Outros metais ativos ainda não têm modelo.')
    if promoter and promoter not in PROMOTERS:
        raise ValueError('Promotor LTFT não suportado: ' + promoter)
    for value, low, high, name in ((temperature, 200, 250, 'Temperatura'), (pressure, 10, 30, 'Pressão'), (ratio, 1.5, 2.2, 'H2/CO')):
        if not math.isfinite(value) or not low <= value <= high:
            raise ValueError(f'{name}: permitido {low} a {high}')


def generate_candidates(metals, promoter='', seed=42):
    validate_config(metals, promoter, 225, 20, 2)
    pool = []
    ratios = range(1, 100) if len(metals) == 2 else [100]
    for share in ratios:
        for loading in range(5, 26):
            for promoter_loading in (range(1, 6) if promoter else [0]):
                for support in SUPPORTS:
                    fractions = {metals[0]: share/100}
                    if len(metals) == 2:
                        fractions[metals[1]] = 1-share/100
                    formula = ''.join(f'{metal}{fraction:.2f}' for metal, fraction in fractions.items())
                    identity = f'{formula}|{loading}|{promoter}|{promoter_loading}|{support}'
                    pool.append({'candidate_id': hashlib.sha256(identity.encode()).hexdigest()[:12],
                        'formula': formula, 'fractions': fractions, 'support': support,
                        'metal_loading_wt_pct': loading, 'promoter': promoter,
                        'promoter_loading_wt_pct': promoter_loading, 'support_wt_pct': 100-loading-promoter_loading})
    # Without a promoter a single metal has only 105 distinct formulations.
    # Never duplicate candidates merely to claim 1000 generated materials.
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(pool), size=min(1000, len(pool)), replace=False)
    return [pool[int(i)] for i in indices]


def descriptors(candidates):
    from pymatgen.core import Composition
    from matminer.featurizers.composition import ElementProperty
    featurizer = ElementProperty.from_preset('magpie')
    labels = featurizer.feature_labels()
    cache = {}
    for candidate in candidates:
        formula = candidate['formula']
        if formula not in cache:
            comp = Composition(formula)
            features = featurizer.featurize(comp)
            # Magpie includes missing properties for some elements: retain NaN explicitly.
            cache[formula] = {**dict(zip(labels, features)), 'mean_atomic_mass': float(comp.weight/comp.num_atoms)}
    return pd.DataFrame([{'candidate_id': c['candidate_id'], **cache[c['formula']]} for c in candidates])


def evaluate(candidate, temperature=225, pressure=20, ratio=2, alpha_override=None):
    fractions = candidate['fractions']
    validate_config(list(fractions), candidate['promoter'], temperature, pressure, ratio)
    base_alpha = sum(PRIORS[m]['alpha']*x for m, x in fractions.items())
    # Declared prior sensitivities; no assertion of quantitative predictive accuracy.
    promoter_shift = .015 if candidate['promoter'] in ('K', 'Na') else 0.0
    support_shift = .02*(SUPPORTS[candidate['support']]-.70)
    alpha = float(np.clip(base_alpha-.001*(temperature-225)-.04*(ratio-2)+.008*math.log(pressure/20)+promoter_shift+support_shift, .50, .98))
    if alpha_override is not None:
        alpha = validate_alpha(alpha_override)
    c5 = tail_fraction(alpha, 5)
    methane = (1-alpha)**2
    activity = sum(PRIORS[m]['activity']*x for m, x in fractions.items())
    phase = sum(PRIORS[m]['phase_score']*x for m, x in fractions.items())
    if len(fractions) == 2:
        phase *= .8  # Mixed-phase uncertainty, not claimed alloy synergy.
    sintering = (temperature-200)/100
    operation = 1-abs(temperature-225)/100
    stability = SUPPORTS[candidate['support']]
    components = dict(activity=activity, C5plus=c5, growth=alpha, phase=phase, stability=stability, operation=operation)
    # Coke/oxidation remain unestimated; do not fabricate a quantitative rate.
    penalty = .10*methane+.07*sintering
    score = max(0.0, sum(WEIGHTS[k]*v for k,v in components.items())-penalty)
    family = next(iter(fractions)) if len(fractions)==1 else 'Co-Fe exploratório'
    product_distribution = distribution(alpha)
    return {**{k:v for k,v in candidate.items() if k!='fractions'}, 'family': family,
        **CHEMICAL_PROFILES[family], 'WGS_extent': None,
        'phase_hypothesis': ' + '.join(PRIORS[m]['phase'] for m in fractions),
        'temperature_C': temperature, 'pressure_bar': pressure, 'H2_CO': ratio,
        'alpha': alpha, 'alpha_origin': 'informado' if alpha_override is not None else 'heurística não calibrada',
        'C5plus_carbon_pct':100*c5, 'CH4_carbon_pct':100*methane,
        **{name+'_carbon_pct':100*fraction for name,fraction in product_distribution['exclusive_groups'].items()},
        'carbon_closure_error':product_distribution['closure_error'],
        'score_LTFT':score, **{'component_'+k:v for k,v in components.items()},
        'penalty_total':penalty, 'CO_conversion_pct':None, 'CO2_selectivity_pct':None,
        'coke_rate':None, 'validation':'Sem calibração experimental',
        'synthesis_route':'Rota proposta: impregnação; definir precursores e tratamentos. '+CHEMICAL_PROFILES[family]['activation'],
        'model_version':MODEL_VERSION}


def run(metals, promoter, output, temperature=225, pressure=20, ratio=2, seed=42, alpha_override=None):
    validate_config(metals, promoter, temperature, pressure, ratio)
    candidates = generate_candidates(metals, promoter, seed)
    descriptor_table = descriptors(candidates)  # Required: failure aborts execution.
    generated = pd.DataFrame([evaluate(c, temperature, pressure, ratio, alpha_override) for c in candidates])
    # All satisfy compositional bounds; the cap is prioritization, not proven thermodynamic viability.
    selected = generated.sort_values(['score_LTFT','candidate_id'], ascending=[False,True]).head(100).copy()
    refined = selected.head(10).copy()
    final = refined.head(2).copy()
    rows = []
    for _, c in refined.iterrows():
        for row in distribution(float(c['alpha']))['rows']:
            rows.append({'candidate_id':c['candidate_id'], **row})
    asf_table = pd.DataFrame(rows)
    product_rows = []
    for _, candidate in refined.iterrows():
        dist = distribution(float(candidate['alpha']))
        for group, fraction in dist['exclusive_groups'].items():
            product_rows.append({'candidate_id':candidate['candidate_id'], 'group':group,
                                 'carbon_pct':100*fraction, 'basis':'carbono nos hidrocarbonetos',
                                 'closure_error':dist['closure_error']})
    products = pd.DataFrame(product_rows)
    out = Path(output)
    out.mkdir(parents=True, exist_ok=True)
    prefix = output_prefix('fischer_tropsch_LTFT', metals, promoter)
    tables = {'gerados':generated, 'selecionados_100':selected, 'refinados_10':refined,
              'prioritarios_2':final, 'descritores_magpie':descriptor_table, 'distribuicao_ASF':asf_table,
              'grupos_produtos':products}
    for name, frame in tables.items():
        frame.to_csv(out/f'{prefix}_{name}.csv', index=False, encoding='utf-8-sig')
    with pd.ExcelWriter(out/f'{prefix}_resultados.xlsx') as writer:
        for name, frame in tables.items(): frame.to_excel(writer, sheet_name=name, index=False)
    metadata = {'version':MODEL_VERSION,'metals':metals,'promoter':promoter,'seed':seed,
        'temperature_C':temperature,'pressure_bar':pressure,'H2_CO':ratio,'alpha_override':alpha_override,
        'counts':{k:len(v) for k,v in tables.items()}, 'weights':WEIGHTS, 'priors':PRIORS,
        'chemical_profiles':CHEMICAL_PROFILES,
        'product_basis':'Fração do carbono dos hidrocarbonetos; não inclui CO/CO2, oxigenados ou coque. C5+ é subtotal, não somar novamente.',
        'chemical_references':['https://doi.org/10.1016/j.cattod.2015.11.005', 'https://www.sciencedirect.com/science/article/pii/S0021951718302550'],
        'alpha_equation':'clip(weighted_alpha - .001*(T-225) - .04*(H2/CO-2) + .008*ln(P/20) + promoter_shift + support_shift, .50, .98)',
        'status':'Triagem heurística LTFT; sem conversão ou produtividade calibradas',
        'descriptor_role':'Magpie e massa atômica registrados para auditoria; sem contribuição aprendida no score',
        'limitations':'ASF condicional aos hidrocarbonetos; fases não calculadas; WGS, coque e oxidação não quantificados; misturas Co-Fe exploratórias'}
    (out/f'{prefix}_resumo.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    report = '<!doctype html><meta charset="utf-8"><title>LTFT</title><style>body{font-family:Arial;margin:32px}table{border-collapse:collapse}td,th{padding:8px;border:1px solid #ccc}</style><h1>Fischer–Tropsch LTFT</h1>'
    report += '<p>'+html.escape(metadata['status'])+'</p><p>'+html.escape(metadata['limitations'])+'</p>'
    report += '<h2>Distribuição de produtos</h2><p>'+html.escape(metadata['product_basis'])+'</p>'+products.to_html(index=False, escape=True)
    report += '<h2>Top 10</h2>'+refined.to_html(index=False, escape=True)+'<h2>Configuração auditável</h2><pre>'+html.escape(json.dumps(metadata, indent=2, ensure_ascii=False))+'</pre>'
    (out/f'{prefix}_relatorio.html').write_text(report, encoding='utf-8')
    return {'tables':tables, 'metadata':metadata, 'output':str(out), 'prefix':prefix}


In [ ]:
resultado = run(metais, promotor, pasta_saida, temperatura_C, pressao_bar, razao_H2_CO, alpha_override=alpha_informado)
display(resultado["tables"]["refinados_10"])
print(resultado["metadata"])
